# 01 — PyNite basics: supports and moment releases

This notebook is intentionally small. We will build the same 6 m beam five times, changing only the **support restraints** and **member-end releases**.

The midpoint carries a 10 kN downward nodal load.

## What you should learn

1. The basic PyNite workflow: create model → add nodes/material/section/members → define supports/releases → add loads → analyze → read results.
2. The difference between a **node support restraint** and a **member-end release**.
3. Why a rotationally free support and a rotational member release can produce the same beam response while representing different connection statements.
4. How the same ideas are represented in CSVs, as in the Z5 workflow.

We use N and mm throughout.

## 1. Read the CSV inputs

The notebook contains almost no hard-coded structural data. Geometry, properties, supports, releases and loads live in `inputs/`.

The five items in `configurations.csv` are **model configurations**, not PyNite load cases. Each configuration changes the stiffness/boundary conditions, so we rebuild the model for each one.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from Pynite import FEModel3D

HERE = Path.cwd()
if not (HERE / "inputs").exists():
    # Convenient when the notebook is launched from the repository root.
    HERE = HERE / "intro_notebooks" / "01_beam_supports_and_releases"

INPUT = HERE / "inputs"

nodes = pd.read_csv(INPUT / "nodes.csv")
materials = pd.read_csv(INPUT / "materials.csv")
sections = pd.read_csv(INPUT / "sections.csv")
members = pd.read_csv(INPUT / "members.csv")
configurations = pd.read_csv(INPUT / "configurations.csv")
supports = pd.read_csv(INPUT / "supports.csv")
releases = pd.read_csv(INPUT / "member_releases.csv")
loads = pd.read_csv(INPUT / "nodal_loads.csv")

display(configurations)
display(nodes)
display(members)

## 2. The PyNite mental model

For this lesson, think of PyNite as seven steps:

1. `FEModel3D()`
2. `add_node(...)`
3. `add_material(...)` and `add_section(...)`
4. `add_member(...)`
5. `def_support(...)` and, where required, `def_releases(...)`
6. `add_node_load(...)` and `add_load_combo(...)`
7. `analyze_linear(...)`

After the analysis, results live on the model's nodes and members.

### The central distinction

A support restraint belongs to a **node**.

For example, `support_RZ=True` says that the node itself cannot rotate about global Z.

A member release belongs to **one end of one member**.

For example, `Rzj=True` in `def_releases` says that the j-end of that member does not transfer its local-z bending moment into the node.

This matters when several members meet at the same node: one member can be released while another remains rigidly connected.

## 3. Why there are apparently "extra" supports

PyNite is a 3D solver, but this is a 2D X-Y problem. Every node therefore has `DZ`, `RX`, and `RY` restrained in the CSV.

Those are **plane-enforcement restraints**. They prevent irrelevant out-of-plane rigid-body modes. For this lesson, focus on:

- `DX`
- `DY`
- `RZ`

At the midpoint, `DX`, `DY`, and `RZ` remain free.

In [ ]:
RELEASE_COLS = [
    "Dxi", "Dyi", "Dzi", "Rxi", "Ryi", "Rzi",
    "Dxj", "Dyj", "Dzj", "Rxj", "Ryj", "Rzj",
]

def as_bool(value):
    return bool(int(value))

def build_and_solve(configuration_id):
    model = FEModel3D()

    # Nodes
    for r in nodes.itertuples(index=False):
        model.add_node(r.node, r.x_mm, r.y_mm, r.z_mm)

    # Materials
    for r in materials.itertuples(index=False):
        model.add_material(
            r.material,
            E=r.E_N_per_mm2,
            G=r.G_N_per_mm2,
            nu=r.nu,
            rho=r.rho,
        )

    # Sections
    for r in sections.itertuples(index=False):
        model.add_section(
            r.section,
            A=r.A_mm2,
            Iy=r.Iy_mm4,
            Iz=r.Iz_mm4,
            J=r.J_mm4,
        )

    # Members
    for r in members.itertuples(index=False):
        model.add_member(
            r.member,
            r.i_node,
            r.j_node,
            r.material,
            r.section,
        )

    # Node supports for this structural configuration
    this_supports = supports.loc[supports.configuration_id == configuration_id]
    for _, r in this_supports.iterrows():
        model.def_support(
            r["node"],
            support_DX=as_bool(r["support_DX"]),
            support_DY=as_bool(r["support_DY"]),
            support_DZ=as_bool(r["support_DZ"]),
            support_RX=as_bool(r["support_RX"]),
            support_RY=as_bool(r["support_RY"]),
            support_RZ=as_bool(r["support_RZ"]),
        )

    # Member-end releases for this configuration.
    # If no row exists, no release is applied.
    this_releases = releases.loc[releases.configuration_id == configuration_id]
    for _, r in this_releases.iterrows():
        kwargs = {name: as_bool(r[name]) for name in RELEASE_COLS}
        model.def_releases(r["member"], **kwargs)

    # Physical load cases
    for r in loads.itertuples(index=False):
        model.add_node_load(
            r.node,
            r.direction,
            r.value,
            case=r.load_case,
        )

    load_case_names = list(dict.fromkeys(loads["load_case"].tolist()))
    model.add_load_combo("Combo 1", {name: 1.0 for name in load_case_names})

    model.analyze_linear(
        log=False,
        check_stability=True,
        check_statics=False,
        sparse=True,
    )
    return model

## 4. Inspect the actual support and release instructions

Before solving anything, inspect the CSV rows as structural statements.

Notice especially that `FF_REL_RIGHT` keeps node B rotationally fixed (`support_RZ = 1`) **and** releases `M2.Rzj`.

That is not contradictory:

- node B is not permitted to rotate;
- member M2 is permitted to rotate relative to node B at that connection;
- therefore M2 cannot deliver a bending moment about that released axis into B.

In [ ]:
display(
    supports[["configuration_id", "node", "support_DX", "support_DY", "support_RZ"]]
)
display(releases)

## 5. Solve all five structural configurations

In [ ]:
models = {
    cid: build_and_solve(cid)
    for cid in configurations["configuration_id"]
}
list(models)

In [ ]:
def combo_value(mapping, combo="Combo 1"):
    return float(mapping.get(combo, 0.0))

def summarize_model(configuration_id, model):
    A = model.nodes["A"]
    MID = model.nodes["MID"]
    B = model.nodes["B"]

    # Reaction values are global.
    RA = combo_value(A.RxnFY)
    RB = combo_value(B.RxnFY)
    MA = combo_value(A.RxnMZ)
    MB = combo_value(B.RxnMZ)

    # Sample the in-plane member bending moment Mz.
    samples = []
    x_offset = 0.0
    for member_name in ["M1", "M2"]:
        m = model.members[member_name]
        Lm = float(m.L())
        xs = np.linspace(0.0, Lm, 41)
        for x in xs:
            samples.append((x_offset + x, float(m.moment("Mz", x, "Combo 1"))))
        x_offset += Lm

    sample_df = pd.DataFrame(samples, columns=["x_mm", "Mz_Nmm"])
    sample_df = sample_df.groupby("x_mm", as_index=False)["Mz_Nmm"].mean()

    return {
        "configuration_id": configuration_id,
        "RA_kN": RA / 1000.0,
        "RB_kN": RB / 1000.0,
        "MA_reaction_kNm": MA / 1e6,
        "MB_reaction_kNm": MB / 1e6,
        "MID_DY_mm": combo_value(MID.DY),
        "max_abs_Mz_kNm": sample_df["Mz_Nmm"].abs().max() / 1e6,
        "moment_samples": sample_df,
    }

summaries = [summarize_model(cid, model) for cid, model in models.items()]
results = pd.DataFrame([
    {k: v for k, v in row.items() if k != "moment_samples"}
    for row in summaries
])

display(results.round(6))

## 6. Compare with elementary beam theory

With:

- (P = 10\,\text{kN})
- (L = 6\,\text{m})
- (E = 210000\,\text{N/mm}^2)
- (I_z = 80\times10^6\,\text{mm}^4)

the expected magnitudes are:

| structural behaviour | (R_A) | (R_B) | fixed-end moment magnitude | midpoint deflection magnitude |
|---|---:|---:|---:|---:|
| fixed-fixed | 5.000 kN | 5.000 kN | 7.500 kN·m at each end | (PL^3/(192EI)) |
| pin-roller | 5.000 kN | 5.000 kN | 0 | (PL^3/(48EI)) |
| fixed-pinned | 6.875 kN | 3.125 kN | 11.250 kN·m at A | (7PL^3/(768EI)) |

The sign of a reported reaction moment depends on axis convention, so first compare **magnitudes and equilibrium**.

In [ ]:
P = 10_000.0
L = 6_000.0
E = 210_000.0
Iz = 80_000_000.0

theory = pd.DataFrame([
    {
        "configuration_id": "FF",
        "RA_theory_kN": 5.000,
        "RB_theory_kN": 5.000,
        "MA_abs_theory_kNm": 7.500,
        "MB_abs_theory_kNm": 7.500,
        "MID_DY_abs_theory_mm": P * L**3 / (192 * E * Iz),
    },
    {
        "configuration_id": "PR",
        "RA_theory_kN": 5.000,
        "RB_theory_kN": 5.000,
        "MA_abs_theory_kNm": 0.000,
        "MB_abs_theory_kNm": 0.000,
        "MID_DY_abs_theory_mm": P * L**3 / (48 * E * Iz),
    },
    {
        "configuration_id": "FP_SUPPORT",
        "RA_theory_kN": 6.875,
        "RB_theory_kN": 3.125,
        "MA_abs_theory_kNm": 11.250,
        "MB_abs_theory_kNm": 0.000,
        "MID_DY_abs_theory_mm": 7 * P * L**3 / (768 * E * Iz),
    },
    {
        "configuration_id": "FF_REL_RIGHT",
        "RA_theory_kN": 6.875,
        "RB_theory_kN": 3.125,
        "MA_abs_theory_kNm": 11.250,
        "MB_abs_theory_kNm": 0.000,
        "MID_DY_abs_theory_mm": 7 * P * L**3 / (768 * E * Iz),
    },
    {
        "configuration_id": "FF_REL_BOTH",
        "RA_theory_kN": 5.000,
        "RB_theory_kN": 5.000,
        "MA_abs_theory_kNm": 0.000,
        "MB_abs_theory_kNm": 0.000,
        "MID_DY_abs_theory_mm": P * L**3 / (48 * E * Iz),
    },
])

comparison = results.merge(theory, on="configuration_id")
comparison["MA_abs_PyNite_kNm"] = comparison["MA_reaction_kNm"].abs()
comparison["MB_abs_PyNite_kNm"] = comparison["MB_reaction_kNm"].abs()
comparison["MID_DY_abs_PyNite_mm"] = comparison["MID_DY_mm"].abs()

display(comparison[[
    "configuration_id",
    "RA_kN", "RA_theory_kN",
    "RB_kN", "RB_theory_kN",
    "MA_abs_PyNite_kNm", "MA_abs_theory_kNm",
    "MB_abs_PyNite_kNm", "MB_abs_theory_kNm",
    "MID_DY_abs_PyNite_mm", "MID_DY_abs_theory_mm",
]].round(6))

## 7. Plot the bending-moment diagrams

These curves make the release effect visible.

Look for two pairs:

- `FP_SUPPORT` and `FF_REL_RIGHT`
- `PR` and `FF_REL_BOTH`

Within numerical precision, each pair should overlay. The **structural response** is the same for this beam/load, but the model statement is different.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for row in summaries:
    s = row["moment_samples"]
    ax.plot(
        s["x_mm"] / 1000.0,
        s["Mz_Nmm"] / 1e6,
        label=row["configuration_id"],
    )

ax.axhline(0.0, linewidth=0.8)
ax.set_xlabel("x along beam [m]")
ax.set_ylabel("PyNite local Mz [kN·m]")
ax.set_title("Bending moment comparison")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

## 8. What exactly did the release do?

Focus on the right-hand end.

### `FP_SUPPORT`

At node B:

- `support_DY = 1`: vertical translation is blocked;
- `support_RZ = 0`: the **node is allowed to rotate**;
- M2 has no member release.

So M2 remains rigidly connected to B, but B itself rotates freely. A nonzero end moment cannot develop there.

### `FF_REL_RIGHT`

At node B:

- `support_DY = 1`;
- `support_RZ = 1`: the **node itself cannot rotate**;
- M2 has `Rzj = 1`: M2's rotational connection to B is released.

So B is fixed, but M2 does not share B's RZ constraint. Again, M2 cannot develop/transmit an end moment about that released axis.

For this simple beam, both paths produce the same in-plane beam response.

In a tower joint with several members, they are **not generally interchangeable**. A support changes the node for every connected member; a member release changes only the selected member end.

## 9. A useful equilibrium check

For every configuration:

[
R_A + R_B \approx 10\text{ kN}
]

because the applied vertical load is 10 kN downward.

For the fixed-pinned behaviour:

[
R_A = \frac{11P}{16}=6.875\text{ kN},
\qquad
R_B = \frac{5P}{16}=3.125\text{ kN}.
]

Moment equilibrium then requires a fixed-end reaction-moment magnitude:

[
|M_A| = \frac{3PL}{16}=11.25\text{ kN·m}.
]

This is a good habit with FEA: **do not treat the solver as the proof.** Check equilibrium and simple limiting cases first.

In [ ]:
equilibrium = results[["configuration_id", "RA_kN", "RB_kN"]].copy()
equilibrium["RA_plus_RB_kN"] = equilibrium["RA_kN"] + equilibrium["RB_kN"]
equilibrium["vertical_error_kN"] = equilibrium["RA_plus_RB_kN"] - 10.0
display(equilibrium.round(9))

## 10. Exercises — edit the CSVs, not the Python

### Exercise A — horizontally restrain the roller

In `supports.csv`, change B's `support_DX` from 0 to 1 for `PR`.

With only a vertical load, almost nothing should change.

Then add a horizontal `FX` load at `MID` in `nodal_loads.csv`. Now the choice matters.

### Exercise B — remove the right release

Delete the `FF_REL_RIGHT,M2,...,Rzj=1` row from `member_releases.csv`.

The configuration then becomes an ordinary fixed-fixed beam. Watch the reactions and end moments jump to the `FF` result.

### Exercise C — release the midpoint by mistake

Try releasing M1's `Rzj` or M2's `Rzi` at `MID`.

That creates an internal hinge at the beam midpoint. This is different from an end support hinge and gives a different structure.

### Exercise D — connect another member to B

That is the next conceptual step for tower modelling. A release on M2 does **not** automatically release the new member. A node restraint, by contrast, acts on the node shared by all members.

---

The main lesson to carry back to Z5 is:

> **Node restraints describe what the joint itself may do. Member releases describe what a particular member is allowed to transmit into that joint.**
